In [253]:
import pandas as pd
import plotly.express as px
import numpy as np
import streamlit as st


coffee_data = pd.read_csv('ProdutosAutorizadosABIC.csv', sep=';') # lendo os dados

In [254]:
# Padronizar colunas com minúsculas, sem acento e sem espaços
coffee_data.columns = coffee_data.columns.str.lower().str.replace(' ', '_').str.replace('ç', 'c').str.replace('ã', 'a').str.replace('í', 'i')

In [255]:
coffee_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1616 entries, 0 to 1615
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   empresa          1616 non-null   str  
 1   estado           1616 non-null   str  
 2   produto          1616 non-null   str  
 3   tipo_do_produto  1616 non-null   str  
 4   certificacao     1616 non-null   str  
 5   tipo_simbolo     1616 non-null   str  
dtypes: str(6)
memory usage: 203.5 KB


In [256]:
coffee_data.head(10)

,empresa,estado,produto,tipo_do_produto,certificacao,tipo_simbolo
0,A & A IND. E COM. DE PRODS. ALIMENTICIOS LTDA,GO,CRISTAL DU PURO,TM,"Pureza, Qualidade",Superior
1,A & A IND. E COM. DE PRODS. ALIMENTICIOS LTDA,GO,CRISTAL DU PURO EXTRA FORTE,TM,"Pureza, Qualidade",Extraforte
2,A B M IND. E COM. CAFE LTDA.,PR,BASA,TM,"Pureza, Qualidade",Tradicional
3,A B M IND. E COM. CAFE LTDA.,PR,BASA EXPRESSO,GR,"Pureza, Qualidade",Gourmet
4,A B M IND. E COM. CAFE LTDA.,PR,COCARI EXPRESSO GOURMET,GR,"Pureza, Qualidade",Gourmet
5,A B M IND. E COM. CAFE LTDA.,PR,COCARI GOURMET,TM,"Pureza, Qualidade",Gourmet
6,A B M IND. E COM. CAFE LTDA.,PR,COCARI TRADICIONAL,TM,"Pureza, Qualidade",Tradicional
7,A B M IND. E COM. CAFE LTDA.,PR,ESTORIL,TM,"Pureza, Qualidade",Tradicional
8,A B M IND. E COM. CAFE LTDA.,PR,LOVAT EXTRA FORTE,TM,"Pureza, Qualidade",Extraforte
9,A. ABRANTES GADELHA CIA,PB,FREI DAMIAO,TM,"Pureza, Qualidade",Tradicional


In [257]:
coffee_data['tipo_simbolo'].unique()

<ArrowStringArray>
[      'Superior',     'Extraforte',    'Tradicional',        'Gourmet',
       'Especial',  'Intensidade 8', 'Intensidade 10',  'Intensidade 6',
  'Intensidade 9',  'Intensidade 7',  'Intensidade 5',  'Intensidade 4']
Length: 12, dtype: str

## Cafés Especiais
- Somente MG, ES e PR produzem os chamados cafés "Especiais", sendo a predominância mineira.  
Esta categoria é considerada a melhor entre todas.

In [258]:
especial_por_estado = coffee_data['estado'][coffee_data['tipo_simbolo'] == 'Especial'].value_counts()
fig = px.bar(especial_por_estado, title='Número de cafés "Especial" por Estado Brasileiro', labels={'Estado': 'Estados Brasileiros', 'value': 'Quantitade de Rótulos'})
fig.update_traces(marker_color='orange')

## Cafés Gourmet
- Maioria dos estados brasileiros possuem produção, sendo a produção predominantemente paulista e mineira.  
Esta categoria é considerada a segunda melhor entre todas.

In [259]:
gourmet_por_estado = coffee_data['estado'][coffee_data['tipo_simbolo'] == 'Gourmet'].value_counts()
fig = px.bar(gourmet_por_estado, title='Número de cafés "Gourmet" por Estado Brasileiro', labels={'estado': 'Estados Brasileiros', 'value': 'Quantitade de Rótulos'})
fig.update_traces(marker_color='blue')


## Cafés Superiores
- Mesmo comportamento dos cafés Gourmet, porém com menor número total de cafés.  
Esta categoria é considerada a terceira melhor entre todas.

In [260]:
superior_por_estado = coffee_data['estado'][coffee_data['tipo_simbolo'] == 'Superior'].value_counts()
fig = px.bar(superior_por_estado, title='Número de cafés "Superior" por Estado Brasileiro', labels={'estado': 'Estados Brasileiros', 'value': 'Quantitade de Rótulos'})
fig.update_traces(marker_color='grey')

### Produção por Estado
- Percebemos que São Paulo e Minas dominam a produção cafeicultora de qualidade (Especial, Gourmet e Superior) do país, sendo que MG tem a maior produção de cafés Especiais (18 rótulos).
- Curioso verificar que São Paulo não produz cafés Especiais, talvez por alguma limitação do tipo de solo ou altitude.
- SP e MG também dominam, com folga, a produção total de café somando todos os tipos.

In [261]:
todos_por_estado = coffee_data['estado'].value_counts()
fig = px.bar(todos_por_estado, title='Número total de cafés por Estado Brasileiro', labels={'estado': 'Estados Brasileiros', 'value': 'Quantitade de Rótulos'})
fig.update_traces(marker_color='red')

## Tipos de apresentação de Cafés
- Produtos são divididos em 3 tipos: Torrados e moídos (TM), Grãos (GR) e Cápsulas (CAP)
- 76,6% de todos os cafés são TM, 20,4% GR e apenas 3% cápsulas
- Enquanto vemos muito marketing sobre produtos em cápsula, a verdade é que a vasta maioria do consumo é moído.

In [262]:
# Verificando proporção de cafés torrados (TM), em grãos (GR) e cápsulas (CAP)

apresentacao_cafes = coffee_data['tipo_do_produto'].value_counts().reset_index()
apresentacao_cafes.rename(columns={'count': 'quantidade'}, inplace=True)

# Criando nova coluna 'percentual' e calculando proporções
apresentacao_cafes['percentual'] = (apresentacao_cafes['quantidade'] / apresentacao_cafes['quantidade'].sum()).round(3)
apresentacao_cafes

,tipo_do_produto,quantidade,percentual
0,TM,1238,0.766
1,GR,329,0.204
2,CAP,49,0.030


## Diversidade de categorias
- São Paulo mostra que tem a maior diversidade de categorias de cafés
- Minas Gerais aparece em 5º lugar, mesmo sendo o 2 maior produtor.

In [263]:
diversidade_categ_cafe = coffee_data.groupby('estado')['tipo_simbolo'].nunique().sort_values(ascending=False)

fig = px.bar(diversidade_categ_cafe, labels={'estado': 'Estados Brasileiros', 'value': 'Quantidade de estilos de café únicos'}, title='Quantidade de diferentes categorias de café por estado brasileiro')
fig.show()

In [264]:
sp_categ_cafe = coffee_data.query('estado == "SP"').groupby(by='tipo_simbolo')['produto'].count().sort_values(ascending=False)

fig = px.bar(sp_categ_cafe, labels={'tipo_simbolo': 'Categorias de Café', 'value': 'Quantidade de Rótulos'}, title='Número de Rótulos por categoria no estado de SP')
fig.update_traces(marker_color='black')

In [265]:
mg_categ_cafe = coffee_data.query('estado == "MG"').groupby(by='tipo_simbolo')['produto'].count().sort_values(ascending=False)

fig = px.bar(mg_categ_cafe, labels={'tipo_simbolo': 'Categorias de Café', 'value': 'Quantidade de Rótulos'}, title='Número de Rótulos por categoria no estado de MG')
fig.update_traces(marker_color='red')

## Categorias Predominantes em SP e MG
- Ambos os estados tem a predominância de rótulos da categoria "Tradicional", que é uma categoria qualidade mediana, e são acompanhados em ambos pela categoria Gourmet
- Isso mostra uma certa dicotomia no perfil de consumo de cafés mineiros e paulistas: enquanto uma parte consume café tradicional (mais barato) a outra parte consome café gourmet (mais caro)
- Em MG o 3º lugar vem com o café "Extraforte", o que é uma surpresa, pois é um café de qualidade baixa, esperando-se uma colocação melhor do café "Superior"


In [273]:
coffee_data.groupby(by='tipo_simbolo')['empresa'].value_counts()
    

tipo_simbolo  empresa                                                   
Especial      CAFE TRES CORACOES S/A                                        10
              CAFFE BRASILIANO DI CHIARA LTDA                                8
              CAFE FAZENDA SERTAOZINHO LTDA.                                 4
              COOP. REGIONAL DE CAFEICULTORES EM GUAXUPE LTDA. - COOXUPE     2
              RUBENS LOS JUNIOR CAFE AGROINDUSTRIA LTDA                      2
                                                                            ..
Tradicional   TORREF. DE CAFE BENEDETTI LTDA ME.                             1
              TORREFACAO BRASIL LTDA.                                        1
              TORREFACAO TERRAS DE SANTANA LTDA                              1
              TREVISANE & TREVISANE LTDA                                     1
              WLE DETTMANN LTDA - ME                                         1
Name: count, Length: 679, dtype: int64